In another notebook, I’ve shown how to build sequences of regular dates (schedules, in QuantLib lingo) and I hinted that they could be used in turn to build sequences of coupons. In this notebook, we’ll do just that; and furthermore, we’ll see how to use those coupons to build a few simple bonds.

In [1]:
import pandas as pd
import QuantLib as ql

ql.Settings.instance().evaluationDate=ql.Date(15, ql.January, 2024)

<h2>A simple fixed-rate bond</h2>
the understanding is that the first date in the schedule is the start of the first coupon; the second date is both the end of the first coupon and the start of the second; the third date is the end of the second coupon and the start of the third; until we get to the last date, which is the end of the last coupon.

In [2]:
schedule = ql.MakeSchedule(
    effectiveDate=ql.Date(8, ql.February, 2021),
    terminationDate=ql.Date(8, ql.February, 2026),
    frequency=ql.Semiannual,
    calendar=ql.TARGET(),
    convention=ql.Following,
    backwards=True   
)

The actual task of building the coupons is done by FixedRateLeg, another function provided by the library, which is the next thing we call (after defining some other bits of information such as the day-count convention for accruing the coupon rate). It takes the schedule, to be used as described above, and the other data we need.

In [3]:
day_count = ql.Thirty360(ql.Thirty360.BondBasis)

fixed_coupons = ql.FixedRateLeg(
    schedule=schedule,
    couponRates=[0.03],
    dayCount=day_count,
    nominals=[10_000]
)

The last step, in order to get a working bond instance, is passing the coupons and some additional information to the constructor of the Bond class.

In [4]:
settlement_days=3
issue_date= ql.Date(5, ql.February, 2021)

bond1 = ql.Bond(
    settlement_days,
    ql.TARGET(),
    issue_date,
    fixed_coupons
)

In [6]:
data=[]
for cf in bond1.cashflows():
    c = ql.as_coupon(cf)
    if c is not None:
        data.append((c.date(), c.nominal(), c.rate(), c.amount()))
    else:
        data.append((cf.date(), None, None, cf.amount()))
        #data.append((cf.date(), None, None, cf.amount()))
pd.DataFrame(
    data, columns=['date', 'nominal','rate','amount']
).style.format({'nominal':'{:.0f}', 'rate':'{:.3%}','amount':'{:3f}'})

,date,nominal,rate,amount
0,"August 9th, 2021",10000,3.000%,150.833333
1,"February 8th, 2022",10000,3.000%,149.166667
2,"August 8th, 2022",10000,3.000%,150.000000
3,"February 8th, 2023",10000,3.000%,150.000000
4,"August 8th, 2023",10000,3.000%,150.000000
5,"February 8th, 2024",10000,3.000%,150.000000
6,"August 8th, 2024",10000,3.000%,150.000000
7,"February 10th, 2025",10000,3.000%,151.666667
8,"August 8th, 2025",10000,3.000%,148.333333
9,"February 9th, 2026",10000,3.000%,150.833333


In [10]:
price=ql.BondPrice(98.0, ql.BondPrice.Clean)
bond1.bondYield(price, day_count, ql.Compounded, ql.Semiannual)

0.04021360635757447

<h2>A shortcut to the same bond</h2>
As you might have guessed, a fixed-rate bond is common enough that we have a more convenient way to instantiate it. Instead of the two separate calls to FixedRateLeg and the Bond constructor, we can call the constructor of the derived FixedRateBond class and pass the same information. Displaying the cash flows of the resulting bond would show that they are the same as the ones above.

In [11]:
bond1b=ql.FixedRateBond(
    settlement_days,
    10000.0,
    schedule,
    [0.03],
    day_count
)

data=[]
for cf in bond1b.cashflows():
    c = ql.as_coupon(cf)
    if c is not None:
        data.append((c.date(), c.nominal(), c.rate(), c.amount()))
    else:
        data.append((cf.date(), None, None, cf.amount()))
        #data.append((cf.date(), None, None, cf.amount()))
pd.DataFrame(
    data, columns=['date', 'nominal','rate','amount']
).style.format({'nominal':'{:.0f}', 'rate':'{:.3%}','amount':'{:3f}'})

,date,nominal,rate,amount
0,"August 9th, 2021",10000,3.000%,150.833333
1,"February 8th, 2022",10000,3.000%,149.166667
2,"August 8th, 2022",10000,3.000%,150.000000
3,"February 8th, 2023",10000,3.000%,150.000000
4,"August 8th, 2023",10000,3.000%,150.000000
5,"February 8th, 2024",10000,3.000%,150.000000
6,"August 8th, 2024",10000,3.000%,150.000000
7,"February 10th, 2025",10000,3.000%,151.666667
8,"August 8th, 2025",10000,3.000%,148.333333
9,"February 9th, 2026",10000,3.000%,150.833333


<h2>An amottizing fixed-rate bond</h2>

In [12]:
amortizing_coupons=ql.FixedRateLeg(
    schedule,
    couponRates=[0.03],
    dayCount=day_count,
    nominals=[
        10000.0,
        10000.0,
        8000.0,
        8000.0,
        6000.0,
        6000.0,
        4000.0,
        4000.0,
        2000.0,
        2000.0
    ]
)

In [13]:
bond2 = ql.Bond(
    settlement_days,
    ql.TARGET(),
    issue_date,
    amortizing_coupons
)

In [18]:
data = []
for cf in bond2.cashflows():
    c = ql.as_coupon(cf)
    if c is not None:
        data.append((c.date(), c.nominal(), c.rate(), c.amount()))
    else:
        data.append((cf.date(), None, None, cf.amount()))

pd.DataFrame(
    data, columns=['date', 'nominal','rate','amount']
).style.format({"nominal": "{:.0f}", "rate": "{:.3%}", "amount": "{:.3f}"})  

,date,nominal,rate,amount
0,"August 9th, 2021",10000,3.000%,150.833
1,"February 8th, 2022",10000,3.000%,149.167
2,"February 8th, 2022",nan,nan%,2000.000
3,"August 8th, 2022",8000,3.000%,120.000
4,"February 8th, 2023",8000,3.000%,120.000
5,"February 8th, 2023",nan,nan%,2000.000
6,"August 8th, 2023",6000,3.000%,90.000
7,"February 8th, 2024",6000,3.000%,90.000
8,"February 8th, 2024",nan,nan%,2000.000
9,"August 8th, 2024",4000,3.000%,60.000
